# Test Feature Extraction on Downloaded Audio

**Goal:** Verify librosa feature extraction works on the 900 downloaded .wav files

**Input:** 
- `youtube_download_status.csv` (list of successfully downloaded songs)
- `.wav` files in `data/raw/youtube_audio/`

**Output:** Test results showing features can be extracted

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import librosa
from tqdm.auto import tqdm

## 1. Load Download Status

In [2]:
# Load download status
status_df = pd.read_csv("../data/raw/youtube_download_status.csv")

print(f"Total records: {len(status_df):,}")
print(f"Successful downloads: {(status_df['download_status'] == 'success').sum():,}")

# Filter to successful downloads only
successful = status_df[status_df['download_status'] == 'success'].copy()
print(f"\nReady for feature extraction: {len(successful):,} songs")

successful.head()

Total records: 900
Successful downloads: 899

Ready for feature extraction: 899 songs


,track_id,track_name,artists,youtube_video_id,youtube_url,download_status,file_path,file_size_mb,error_message
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward,lcOJS28FTbY,https://www.youtube.com/watch?v=lcOJS28FTbY,success,../data/raw/youtube_audio/4qPNDBW1i3p13qLCt0Ki...,38.96,NaN
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN,gbh5JEOrwt0,https://www.youtube.com/watch?v=gbh5JEOrwt0,success,../data/raw/youtube_audio/1iJBSr7s7jYXzM8EGcbK...,38.39,NaN
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Kina Grannis,tyoAsprwDRc,https://www.youtube.com/watch?v=tyoAsprwDRc,success,../data/raw/youtube_audio/6lfxq3CG4xtTiEg7opyC...,60.46,NaN
4,5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,a0sxc2jIYD0,https://www.youtube.com/watch?v=a0sxc2jIYD0,success,../data/raw/youtube_audio/5vjLSffimiIP26QG5WcN...,36.87,NaN
5,01MVOl9KtVTNfFiBU9I7dc,Days I Will Remember,Tyrone Wells,8FPksYzZ3_M,https://www.youtube.com/watch?v=8FPksYzZ3_M,success,../data/raw/youtube_audio/01MVOl9KtVTNfFiBU9I7...,39.23,NaN


## 2. Feature Extraction Function (from your working code)

In [3]:
def extract_audio_features(file_path):
    """
    Extract comprehensive audio features using librosa
    """
    y, sr = librosa.load(file_path, sr=None)
    duration = librosa.get_duration(y=y, sr=sr)

    # Tempo
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    tempo = np.asarray(tempo).squeeze()
    tempo = float(tempo) if np.ndim(tempo) == 0 else float(tempo[0])

    # Energy features
    rms = librosa.feature.rms(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]

    # Spectral features
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    spectral_flatness = librosa.feature.spectral_flatness(y=y)[0]

    # Pitch
    f0, voiced_flag, voiced_probs = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7")
    )

    if f0 is not None:
        valid_f0 = f0[~np.isnan(f0)]
    else:
        valid_f0 = np.array([])

    if len(valid_f0) > 0:
        pitch_mean = float(np.mean(valid_f0))
        pitch_std = float(np.std(valid_f0))
        pitch_min = float(np.min(valid_f0))
        pitch_max = float(np.max(valid_f0))
    else:
        pitch_mean = np.nan
        pitch_std = np.nan
        pitch_min = np.nan
        pitch_max = np.nan

    # Onset features
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    onset_frames = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr)
    onset_count = int(len(onset_frames))
    onset_density_per_sec = float(onset_count / duration) if duration > 0 else np.nan

    # Tempogram
    tempogram = librosa.feature.tempogram(onset_envelope=onset_env, sr=sr)

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)

    # Tonnetz
    y_harmonic = librosa.effects.harmonic(y)
    tonnetz = librosa.feature.tonnetz(y=y_harmonic, sr=sr)

    # Build feature dictionary
    features = {
        "sample_rate": sr,
        "duration_sec": float(duration),
        "tempo_bpm": tempo,

        "rms_mean": float(np.mean(rms)),
        "rms_std": float(np.std(rms)),

        "zcr_mean": float(np.mean(zcr)),
        "zcr_std": float(np.std(zcr)),

        "spectral_centroid_mean": float(np.mean(spectral_centroid)),
        "spectral_centroid_std": float(np.std(spectral_centroid)),

        "spectral_bandwidth_mean": float(np.mean(spectral_bandwidth)),
        "spectral_bandwidth_std": float(np.std(spectral_bandwidth)),

        "spectral_rolloff_mean": float(np.mean(spectral_rolloff)),
        "spectral_rolloff_std": float(np.std(spectral_rolloff)),

        "spectral_contrast_mean": float(np.mean(spectral_contrast)),
        "spectral_contrast_std": float(np.std(spectral_contrast)),

        "spectral_flatness_mean": float(np.mean(spectral_flatness)),
        "spectral_flatness_std": float(np.std(spectral_flatness)),

        "pitch_mean": pitch_mean,
        "pitch_std": pitch_std,
        "pitch_min": pitch_min,
        "pitch_max": pitch_max,

        "onset_strength_mean": float(np.mean(onset_env)),
        "onset_strength_std": float(np.std(onset_env)),
        "onset_strength_max": float(np.max(onset_env)) if len(onset_env) > 0 else np.nan,

        "onset_count": onset_count,
        "onset_density_per_sec": onset_density_per_sec,

        "tempogram_mean": float(np.mean(tempogram)),
        "tempogram_std": float(np.std(tempogram)),
        "tempogram_max": float(np.max(tempogram)) if tempogram.size > 0 else np.nan,
    }

    # MFCC mean/std
    for i in range(13):
        features[f"mfcc_{i+1}_mean"] = float(np.mean(mfcc[i]))
        features[f"mfcc_{i+1}_std"] = float(np.std(mfcc[i]))

    # Chroma mean/std
    for i in range(12):
        features[f"chroma_{i+1}_mean"] = float(np.mean(chroma[i]))
        features[f"chroma_{i+1}_std"] = float(np.std(chroma[i]))

    # Tonnetz mean/std
    for i in range(6):
        features[f"tonnetz_{i+1}_mean"] = float(np.mean(tonnetz[i]))
        features[f"tonnetz_{i+1}_std"] = float(np.std(tonnetz[i]))

    return features

## 3. Test on Single File

In [4]:
# Test with first successful download
test_song = successful.iloc[0]

print(f"Testing feature extraction on:")
print(f"  Track: {test_song['track_name']}")
print(f"  Artist: {test_song['artists']}")
print(f"  File: {test_song['file_path']}")
print(f"  Size: {test_song['file_size_mb']} MB")
print(f"\nExtracting features...\n")

test_features = extract_audio_features(test_song['file_path'])

print(f"✓ Success!")
print(f"\nExtracted {len(test_features)} features:")
print(f"  Sample rate: {test_features['sample_rate']} Hz")
print(f"  Duration: {test_features['duration_sec']:.2f} seconds")
print(f"  Tempo: {test_features['tempo_bpm']:.2f} BPM")
print(f"  Pitch (mean): {test_features['pitch_mean']:.2f} Hz")
print(f"  Spectral centroid: {test_features['spectral_centroid_mean']:.2f} Hz")

Testing feature extraction on:
  Track: Ghost - Acoustic
  Artist: Ben Woodward
  File: ../data/raw/youtube_audio/4qPNDBW1i3p13qLCt0Ki3A.wav
  Size: 38.96 MB

Extracting features...

✓ Success!

Extracted 91 features:
  Sample rate: 48000 Hz
  Duration: 212.80 seconds
  Tempo: 104.17 BPM
  Pitch (mean): 130.48 Hz
  Spectral centroid: 2687.18 Hz


## 4. Test on 10 Files

In [5]:
# Test with first 10 songs
test_df = successful.head(10).copy()

print(f"Testing feature extraction on {len(test_df)} songs...\n")

results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        features = extract_audio_features(row['file_path'])
        
        result = {
            'track_id': row['track_id'],
            'track_name': row['track_name'],
            'artists': row['artists'],
            'extraction_success': True,
            'error_message': '',
            **features  # Add all extracted features
        }
        
    except Exception as e:
        result = {
            'track_id': row['track_id'],
            'track_name': row['track_name'],
            'artists': row['artists'],
            'extraction_success': False,
            'error_message': str(e)
        }
    
    results.append(result)

results_df = pd.DataFrame(results)

print(f"\n=== Results ===")
print(f"Successful: {results_df['extraction_success'].sum()}/10")
print(f"Failed: {(~results_df['extraction_success']).sum()}/10")

# Show sample features
if results_df['extraction_success'].sum() > 0:
    print(f"\n=== Sample Features ===")
    display(results_df[
        results_df['extraction_success']
    ][[
        'track_name', 'artists', 'tempo_bpm', 'duration_sec', 
        'pitch_mean', 'spectral_centroid_mean'
    ]])

Testing feature extraction on 10 songs...



  0%|          | 0/10 [00:00<?, ?it/s]


=== Results ===
Successful: 10/10
Failed: 0/10

=== Sample Features ===


,track_name,artists,tempo_bpm,duration_sec,pitch_mean,spectral_centroid_mean
0,Ghost - Acoustic,Ben Woodward,104.166667,212.800000,130.482638,2687.179359
1,To Begin Again,Ingrid Michaelson;ZAYN,144.230769,209.641375,269.428745,2468.136020
2,Can't Help Falling In Love,Kina Grannis,100.446429,330.164542,95.357230,1829.563512
3,Hold On,Chord Overstreet,119.680851,201.340229,84.594006,1981.211937
4,Days I Will Remember,Tyrone Wells,98.684211,214.238917,108.480790,3895.462547
5,Say Something,A Great Big World;Christina Aguilera,140.625000,231.282375,102.326931,1593.906957
6,I'm Yours,Jason Mraz,152.027027,220.972708,101.297093,2363.940305
7,Lucky,Jason Mraz;Colbie Caillat,133.928571,202.303875,123.590973,2228.456798
8,Hunger,Ross Copperman,156.250000,205.594125,84.706191,1760.513691
9,Give Me Your Forever,Zack Tabudlo,100.446429,365.284729,101.769889,2276.817924


## 5. Conclusion

If all 10 tests pass:
✅ Feature extraction works with your downloaded audio files  
✅ File naming with `track_id` works correctly  
✅ Ready to build the combined pipeline!

**Next:** Build `05_full_pipeline_combined.ipynb` that does Match → Download → Extract in one notebook